# Final Broad Annotation of Integrated Atlas

**Steps.**
1. Label each Leiden cluster with a broad cell type, using the predictions and markers.
2. Sub-cluster ambiguous or composite labels and relabel them (applied in order, so a later
   step can refine the output of an earlier one).
3. Remove clusters flagged for exclusion.
4. Add patient IDs, colours, and sanity checks (markers, predictions, composition).

**Output.** Final object with `cell_type` (broad labels) and `ID_2` (patient/sample ID),
raw counts in `X`; figures and summary tables.

In [ ]:
# !pip install --quiet scanpy leidenalg igraph scikit-learn
# %cd <project>/notebooks

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from sklearn.feature_extraction.text import TfidfTransformer

## Config

In [ ]:
input_path = "./../snRNAseq/output/cellbender/cleanup/S14c_scVI_integrated_adata_clean_latent20.h5ad" #decide on latent20

output_path = "./../snRNAseq/output/cellbender/cleanup/scVI_integrated_adata_clean_latent20_major_cell_type_add.h5ad"

figures_output_dir = "./../figures-cellbender/integration/3_annotation/2_clean/1_broad"
tables_output_dir = "./../results/annotation/final_broad"
seed = 0

cluster_key = "leiden_scvi"             # clusters of the integrated object
label_col = "cell_type"                 # final broad label (overwritten)
previous_label_col = "cell_type_previous"   # labels carried in from earlier annotation, kept for comparison
counts_layer = "raw_counts"
condition_col = "group2"

subcluster_qc_cols = ["phase", "scrublet_doublet", "n_genes", "percent_mito", "recist", "stage_at_diagnosis"]
n_marker_genes = 10
run_tfidf = True
tfidf_cells_as_documents = False        # False reproduces the original (fitted on genes x cells)


os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir
sc.settings.verbosity = 1
sc.set_figure_params(dpi=80, figsize=(4, 4))

## 1. Load integrated object

In [ ]:
adata = sc.read(input_path)
adata.uns.pop("log1p", None)
vals = adata.layers[counts_layer][:1000].data
assert np.allclose(vals, np.round(vals)), f"layers['{counts_layer}'] must contain raw counts"
assert not adata.obs["is_doublet"].astype(str).eq("True").any(), "doublets should have been removed upstream"

if label_col in adata.obs and previous_label_col not in adata.obs:
    adata.obs[previous_label_col] = adata.obs[label_col].astype(str)

sc.pl.umap(adata, color=["probabilities", "celltype_predictions"], size=15, save="_labTransfer_combine.pdf")
sc.pl.umap(adata, color=[cluster_key, "sample", "batchseq", "n_genes", "scrublet_score", "percent_mito",
                         condition_col, previous_label_col], color_map="OrRd", ncols=4, wspace=0.3, size=15,
           save="_leiden_qc_combine.pdf")
adata

## 2. Helpers

- `tfidf_markers`: TF-IDF on binarised counts + Wilcoxon ranking (marker discovery).
- `prediction_crosstab`: top reference predictions per group; the label-transfer evidence.
- `check_genes`: drops panel genes absent from the object and reports them.
- `refine`: sub-cluster one set of labels, plot evidence, return new labels (or None).

In [ ]:
def tfidf_markers(adata, groupby, method="wilcoxon", cells_as_documents=tfidf_cells_as_documents):
    """Rank genes per group on TF-IDF-transformed binarised counts (X must hold counts)."""
    X_bin = (sparse.csr_matrix(adata.X) > 0).astype(np.float32)
    tfidf = (TfidfTransformer().fit_transform(X_bin) if cells_as_documents
             else TfidfTransformer().fit_transform(X_bin.T).T)
    bdata = sc.AnnData(X=sparse.csr_matrix(tfidf), obs=adata.obs[[groupby]].copy(), var=adata.var[[]].copy())
    bdata.obs[groupby] = bdata.obs[groupby].astype("category").cat.remove_unused_categories()
    sc.tl.rank_genes_groups(bdata, groupby=groupby, use_raw=False, method=method)
    return bdata


def prediction_crosstab(obs, groupby, top=3):
    """Top predicted labels (fraction of cells) and median probability per group."""
    ct = pd.crosstab(obs[groupby], obs["celltype_predictions"], normalize="index")
    top_labels = {g: "; ".join(f"{k} ({v:.0%})" for k, v in row.sort_values(ascending=False).head(top).items() if v > 0)
                  for g, row in ct.iterrows()}
    summary = pd.DataFrame({"n_cells": obs[groupby].value_counts(), "top_predictions": pd.Series(top_labels)})
    summary["median_probability"] = obs.groupby(groupby, observed=True)["probabilities"].median()
    return summary


def check_genes(panels, var_names, label=""):
    """Keep panel genes present in var_names; report the rest."""
    kept, missing = {}, []
    for name, genes in panels.items():
        present = [g for g in dict.fromkeys(genes) if g in var_names]
        missing += [g for g in genes if g not in var_names]
        if present:
            kept[name] = present
    if missing:
        print(f"{label}: not found, skipped -> {missing}")
    return kept


def refine(adata, name, parents, markers, mapping=None, default=None, expected_n_clusters=None):
    """Sub-cluster cells currently labelled `parents`, save evidence, return new labels (None if explore only)."""
    sub = adata[adata.obs[label_col].isin(parents)].copy()
    if sub.n_obs <= subcluster_n_neighbors:
        raise ValueError(f"{name}: {sub.n_obs} cells, too few to sub-cluster")
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)
    sc.pp.neighbors(sub, n_neighbors=subcluster_n_neighbors, use_rep=subcluster_use_rep, random_state=seed)
    sc.tl.umap(sub, random_state=seed)
    sc.tl.leiden(sub, key_added="leiden_sub", resolution=subcluster_resolution, flavor="leidenalg", random_state=seed)
    n_clusters = sub.obs["leiden_sub"].nunique()
    print(f"{name}: {sub.n_obs} cells, {n_clusters} sub-clusters")

    sc.pl.umap(sub, color=["leiden_sub", label_col, "celltype_predictions"] + subcluster_qc_cols, ncols=4,
               wspace=0.4, size=15, color_map="OrRd", legend_fontsize="xx-small", save=f"_{name}_subclusters_qc.pdf")
    sc.tl.rank_genes_groups(sub, "leiden_sub", method="logreg", key_added="logreg", max_iter=1000)
    sc.pl.rank_genes_groups_dotplot(sub, n_genes=n_marker_genes, key="logreg", groupby="leiden_sub",
                                    standard_scale="var", dendrogram=False, save=f"{name}_logreg.pdf")
    if run_tfidf:
        counts = sc.AnnData(X=adata[sub.obs_names].layers[counts_layer], obs=sub.obs[["leiden_sub"]], var=adata.var[[]])
        sc.pl.rank_genes_groups_dotplot(tfidf_markers(counts, "leiden_sub"), n_genes=n_marker_genes,
                                        dendrogram=False, standard_scale="var", save=f"{name}_tfidf.pdf")
    panels = check_genes(markers, sub.var_names, name)
    sc.pl.dotplot(sub, panels, groupby="leiden_sub", standard_scale="var", save=f"{name}_marker_panels.pdf")
    genes = list(dict.fromkeys(g for v in panels.values() for g in v))
    sc.pl.umap(sub, color=["leiden_sub"] + genes, color_map="OrRd", size=25, ncols=4, wspace=0.1,
               legend_loc="on data", legend_fontsize="small", save=f"_{name}_markers.pdf")

    summary = prediction_crosstab(sub.obs, "leiden_sub")
    new_labels = None
    if mapping is not None:
        clusters = sub.obs["leiden_sub"].astype(str)
        if expected_n_clusters is not None and n_clusters != expected_n_clusters:
            raise ValueError(f"{name}: {n_clusters} sub-clusters, mapping written for {expected_n_clusters}")
        unmapped = sorted(set(clusters) - set(mapping), key=int)
        if unmapped and default is None:
            raise ValueError(f"{name}: sub-clusters without a label: {unmapped}")
        new_labels = clusters.map(lambda c: mapping.get(c, default))
        summary["new_label"] = summary.index.map(lambda c: mapping.get(c, default))
        sub.obs["new_label"] = new_labels.astype("category")
        sc.pl.umap(sub, color="new_label", size=25, legend_fontsize="x-small", legend_fontoutline=True,
                   title=f"{name}: new labels", save=f"_{name}_annotated.pdf")
    summary.to_csv(f"{tables_output_dir}/{name}_subcluster_summary.csv")
    display(summary)
    return new_labels

## 3. Broad labels per cluster

The table shows, for each Leiden cluster, the top reference predictions next to the
assigned broad label.

In [ ]:
# --- step 1: broad label per Leiden cluster ---
broad_map = {
    "0": "Alveolar cell type I", "1": "Epithelial+Tumor cells", "2": "Alveolar cell type II",
    "3": "Epithelial+Tumor cells", "4": "Epithelial+Tumor cells", "5": "Myeloid",
    "6": "Epithelial+Tumor cells", "7": "Fibroblast", "8": "Lymphoid", "9": "Epithelial+Tumor cells",
    "10": "mix", "11": "Alveolar cell type II", "12": "Goblet_club cell", "13": "Non-adeno",
    "14": "Epithelial+Tumor cells", "15": "Endothelial", "16": "Ciliated epithelial",
    "17": "Alveolar cell type I", "18": "Non-adeno", "19": "Lymphoid", "20": "Epithelial+Tumor cells",
    "21": "Epithelial+Tumor cells", "22": "Myeloid", "23": "mix",
}

unmapped = set(adata.obs[cluster_key].astype(str)) - set(broad_map)
assert not unmapped, f"clusters without a broad label: {sorted(unmapped)}"

cluster_summary = prediction_crosstab(adata.obs, cluster_key)
cluster_summary["broad_label"] = cluster_summary.index.map(broad_map)
cluster_summary.to_csv(f"{tables_output_dir}/cluster_prediction_summary.csv")
cluster_summary

## 4. Sub-cluster refinement and exclusion

Starts from `broad_map` every time, so the cell can be re-run safely. Refinements run in
order on the current labels; entries with `mapping=None` only produce review plots.

In [ ]:
# --- step 2: sub-cluster refinements, applied in order on the current labels ---
# parents: labels to pool | mapping: sub-cluster -> label (None = explore only)
# default: label for unmapped sub-clusters | expected_n_clusters: guard against drifting Leiden IDs
subcluster_use_rep = "X_scVI_n_latent_10"   # latent used for sub-clustering (as in the original run)
subcluster_n_neighbors = 30
subcluster_resolution = 0.6
alveolar_airway_markers = {
    "AT1": ["AGER"], "AT2": ["SFTPC", "SFTPD"],
    "Transitional (DATP)": ["KRT8", "CLDN4"],
    "Basal": ["KRT17", "KRT5", "TP63", "LAMC2", "KRT14"],
    "Club": ["SOX9", "CASC15", "BICC1", "HOPX", "SCGB3A2", "SCGB3A1"],
    "Goblet": ["SCGB1A1", "AGR2", "MUC5B", "MUC5AC", "BPIFB1"],
}


refinements = [
    dict(name="mix", parents=["mix"],
         markers={**alveolar_airway_markers,
                  "Tumour": ["NAPSA", "EPCAM", "NKX2-1", "CEACAM5"],
                  "Myeloid": ["PPARG", "CD68", "CD14", "FCGR3A"],
                  "Endothelial": ["CD34", "PECAM1", "VEGFA"]},
         mapping={"0": "Epithelial+Tumor cells", "1": "Endothelial", "2": "Endothelial",
                  "3": "Epithelial+Tumor cells", "4": "Endothelial", "5": "Endothelial",
                  "6": "Epithelial+Tumor cells", "7": "Endothelial", "8": "Endothelial",
                  "9": "Epithelial+Tumor cells", "10": "regress", "11": "Epithelial+Tumor cells"},
         expected_n_clusters=12),
    dict(name="alveolar_goblet_club", parents=["Alveolar cell type I", "Alveolar cell type II", "Goblet_club cell"],
         markers={**alveolar_airway_markers, "Other": ["GDF15", "LDLRAD4", "IL1B", "IDO1"]},
         mapping={"0": "Alveolar cell type II", "1": "Alveolar cell type I", "2": "Alveolar cell type I",
                  "3": "Alveolar cell type I", "4": "Alveolar cell type II", "5": "Alveolar cell type I",
                  "6": "Alveolar cell type II", "7": "Alveolar cell type II", "8": "Alveolar cell type II",
                  "9": "Transitional", "10": "Goblet_club cell"},
         expected_n_clusters=11),
    dict(name="goblet_club", parents=["Goblet_club cell"],
         markers={**alveolar_airway_markers, "Other": ["GDF15", "EGFR", "MET"]},
         mapping={"3": "Goblet cell", "6": "Goblet cell"}, default="Club cell",
         expected_n_clusters=9),
    # review only: checks for B/plasma and myeloid contamination; states annotated in compartment notebooks
    dict(name="lymphoid", parents=["Lymphoid"],
         markers={"Myeloid": ["CD14"], "B cell": ["MEF2C", "AFF3"], "Plasma": ["SDC1", "MZB1"]},
         mapping=None),
    dict(name="myeloid", parents=["Myeloid"],
         markers={"Monocyte/macrophage": ["CD14", "MERTK", "STAB1", "CD36"],
                  "Alveolar macrophage": ["INHBA", "PPARG"], "DC": ["CLEC4A", "BIRC3"]},
         mapping=None),
]

In [ ]:

adata.obs[label_col] = adata.obs[cluster_key].astype(str).map(broad_map).astype("category")
for spec in refinements:
    print(f"=== {spec['name']} ===")
    new_labels = refine(adata, **spec)
    if new_labels is not None:
        labels = adata.obs[label_col].astype(str)
        labels.loc[new_labels.index] = new_labels
        adata.obs[label_col] = labels.astype("category")


# --- step 3: labels removed from the final object ---
remove_labels = ["regress"]

removed = adata.obs[label_col].isin(remove_labels)
print(f"removing {removed.sum()} cells labelled {remove_labels}")
adata = adata[~removed].copy()
adata.obs[label_col] = adata.obs[label_col].cat.remove_unused_categories()
adata.obs[label_col].value_counts()

## 5. Patient IDs, colours and final checks

- Every sample must have an `ID_2`; colours must cover every label.
- Previous vs final labels: where the new annotation departs from earlier labels.
- Marker dotplot and prediction summary per final label.
- Composition per patient (table + stacked bars).

In [ ]:
# --- step 4: patient/sample ID (ID_2), colours, sanity markers ---
id_col = "ID_2"
sample_to_id = {
    # treatment-naive tumour / paired adjacent normal
    "A074LCG": "TN001", "A074LCGAN": "TN001_AN", "A150LCG": "TN002", "A150LCGAN": "TN002_AN",
    "A157LCG": "TN003", "A157LCGAN": "TN003_AN", "A111LCG": "TN004", "A111LCGAN": "TN004_AN",
    "A3944LCG": "TN005", "A3944LCGAN": "TN005_AN", "A4177LCG": "TN006", "A4177LCGAN": "TN006_AN",
    "A4243LCG": "TN007", "A4243LCGAN": "TN007_AN", "A161LCG": "TN008", "A161LCGAN": "TN008_AN",
    "A3808LCG": "TN009", "A3808LCGAN": "TN009_AN",
    # on treatment
    "A168LCG": "P001", "A239LCG": "P002", "A315LCG": "P003", "A511LCG": "P005", "A425LCG": "P006",
    "A514LCG": "P008", "A524LCG": "P009", "A554LCG": "P011", "A584LCG": "P012", "A644LCG": "P013",
    # TKI resistant
    "A787LCG": "R001", "A1089LCG": "R002", "A2062LCG": "R003", "A2503LCG": "R004", "A2503LCG_1": "R004",
    "A2055LCG": "R005", "IP3327LCG": "R006", "IP2991LCG": "R007", "IP3586LCG": "R008",
    "IP2612LCG": "R009", "IP2129LCG": "R010", "IP2844LCG": "R011", "A1037LCG": "R012",
    # other cohorts
    "A320LCG": "D001", "A2486LCG": "D002", "A683LCG": "D003", "IP1716LCG": "E001",
    "A1633LCG": "O001", "A1633LCG_1": "O001",
}

cell_type_colors = {
    "Epithelial+Tumor cells": "#023fa5", "Non-adeno": "#4a6fe3",
    "Alveolar cell type I": "#7d87b9", "Alveolar cell type II": "#bec1d4", "Transitional": "#e996c3",
    "Club cell": "#8e063b", "Goblet cell": "#d6bcc0", "Ciliated epithelial": "#bb7784",
    "Endothelial": "#93019f", "Fibroblast": "#e07b91", "Myeloid": "#d33f6a", "Lymphoid": "#ef9708",
}

In [ ]:
missing = set(adata.obs["sample"]) - set(sample_to_id)
assert not missing, f"samples without an {id_col}: {sorted(missing)}"
adata.obs[id_col] = adata.obs["sample"].map(sample_to_id).astype("category")

labels = list(adata.obs[label_col].cat.categories)
no_color = [l for l in labels if l not in cell_type_colors]
assert not no_color, f"labels without a colour: {no_color}"
adata.uns[f"{label_col}_colors"] = [cell_type_colors[l] for l in labels]   # list in category order (scanpy convention)

sc.pl.umap(adata, color=label_col, size=10, legend_fontsize="x-small", legend_fontoutline=True,
           save="_annotated_major_cell_type_combine.pdf")
sc.pl.umap(adata, color=[condition_col, id_col], size=10, wspace=0.5, legend_fontsize="xx-small",
           save="_condition_patient_combine.pdf")

if previous_label_col in adata.obs:
    transitions = pd.crosstab(adata.obs[previous_label_col], adata.obs[label_col])
    transitions.to_csv(f"{tables_output_dir}/previous_vs_final_labels.csv")
    display(transitions)

In [ ]:
broad_markers = {
    "Epithelial": ["EPCAM", "KRT8", "NKX2-1", "NAPSA"], "AT1": ["AGER", "RTKN2"],
    "AT2": ["SFTPC", "SFTPB", "LAMP3"], "Transitional": ["CLDN4", "KRT17"],
    "Club": ["SCGB3A1", "SCGB3A2"], "Goblet": ["MUC5B", "BPIFB1"], "Ciliated": ["FOXJ1", "CAPS"],
    "Endothelial": ["PECAM1", "VWF"], "Fibroblast": ["COL1A1", "PDGFRA"],
    "Myeloid": ["CD68", "CD14"], "Lymphoid": ["PTPRC", "CD3E", "MS4A1", "MZB1"],
}

In [ ]:
marker_panel = check_genes(broad_markers, adata.var_names, "broad markers")
tmp = adata.copy()
sc.pp.normalize_total(tmp, target_sum=1e4)
sc.pp.log1p(tmp)
sc.pl.dotplot(tmp, marker_panel, groupby=label_col, standard_scale="var", save="final_cell_type_markers.pdf")
del tmp

final_summary = prediction_crosstab(adata.obs, label_col)
final_summary.to_csv(f"{tables_output_dir}/final_label_prediction_summary.csv")
final_summary

In [ ]:
def proportion_barplot(adata, group_col, fname):
    """Stacked horizontal bars: % of each label per group."""
    props = pd.crosstab(adata.obs[group_col], adata.obs[label_col], normalize="index") * 100
    props = props.iloc[::-1]
    ax = props.plot.barh(stacked=True, edgecolor="none", width=0.85, figsize=(4, 0.25 * len(props) + 1),
                         color=[cell_type_colors[c] for c in props.columns], grid=False)
    ax.set_xlim(0, 100)
    ax.set_xlabel(f"{label_col} (%)")
    ax.legend(bbox_to_anchor=(1, 1), frameon=False, fontsize=8)
    plt.savefig(f"{figures_output_dir}/{fname}", bbox_inches="tight", dpi=300)
    plt.show()


counts = pd.crosstab(adata.obs[id_col], adata.obs[label_col], margins=True)
counts.to_csv(f"{tables_output_dir}/final_cell_counts_per_{id_col}.csv")
pd.crosstab(adata.obs[condition_col], adata.obs[label_col], margins=True).to_csv(
    f"{tables_output_dir}/final_cell_counts_per_{condition_col}.csv")
proportion_barplot(adata, "sample", f"barplot_combine_sample_proportions_{label_col}.pdf")
proportion_barplot(adata, id_col, f"barplot_combine_{id_col}_proportions_{label_col}.pdf")
counts

## 6. Save final object (X = raw counts)

In [ ]:
adata.X = adata.layers[counts_layer].copy()
adata.uns.pop("log1p", None)
adata.write(output_path)
print(f"final object: {adata.n_obs} cells, {adata.obs[label_col].nunique()} broad labels, "
      f"{adata.obs[id_col].nunique()} patient samples")